## 🎯 Learning Objectives
* Understand the core concept of word embeddings and their purpose in NLP.
* Implement a simplified Word2Vec-like model (CBOW) using PyTorch.
* Train an embedding layer to learn vector representations of words.
* Extract learned word embeddings from a PyTorch model.
* Apply dimensionality reduction techniques (t-SNE or UMAP) to visualize high-dimensional embeddings.
* Interpret the spatial relationships of words in a 2D embedding space.


# DL02-L04 Exercise: Train a Word Embedding and Visualize It

Welcome to this hands-on exercise! In the previous lessons, we explored the theoretical foundations of word embeddings and their significance in modern NLP. Now, it's time to put that knowledge into practice.

**Objective:** Your task is to implement a simplified Word2Vec-like model (specifically, a Continuous Bag-of-Words, or CBOW, architecture) using PyTorch. You will train this model on a small, mock text corpus to learn word embeddings, and then visualize these embeddings in a 2D space using dimensionality reduction techniques.

This exercise will solidify your understanding of how word embeddings are learned and how they capture semantic relationships between words.

## Task Breakdown:

1.  **Data Preparation:**
    *   Tokenize the provided mock corpus.
    *   Build a vocabulary and create mappings from words to indices and vice-versa.
    *   Generate CBOW-style training pairs (context words -> target word).

2.  **Model Implementation:**
    *   Define a simple PyTorch neural network that takes context word indices as input, averages their embeddings, and predicts the target word.
    *   Use `torch.nn.Embedding` for the embedding layer.

3.  **Training Loop:**
    *   Implement a training loop to iterate over the dataset for a specified number of epochs.
    *   Use an appropriate loss function (e.g., `nn.NLLLoss` or `nn.CrossEntropyLoss`) and an optimizer (e.g., `Adam` or `SGD`).

4.  **Embedding Extraction:**
    *   After training, extract the learned word vectors from your model's embedding layer.

5.  **Visualization:**
    *   Apply a dimensionality reduction technique (like t-SNE or UMAP) to project the high-dimensional word embeddings into a 2D space.
    *   Plot these 2D points using `matplotlib`, annotating each point with its corresponding word.

## Requirements:

*   Use PyTorch for all model definitions and training.
*   Ensure your code is well-commented and easy to understand.
*   The visualization should clearly show word relationships.
*   Handle potential edge cases (e.g., words not in vocabulary, empty contexts).

## Evaluation Criteria:

*   **Correctness:** Does the model train without errors? Are the embeddings extracted correctly?
*   **Functionality:** Does the visualization generate a plot?
*   **Quality of Embeddings:** Do semantically similar words appear close to each other in the visualization (even with a small dataset, some patterns should emerge)?
*   **Code Clarity:** Is the code clean, readable, and well-commented?
*   **Adherence to PyTorch Best Practices:** Proper use of `nn.Module`, `optim`, `loss` functions, and tensor operations.

Good luck!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE # A common choice for visualization
# import umap # For 2026, UMAP is often preferred for larger datasets, but TSNE is simpler for this exercise.
import numpy as np
import random

# --- Configuration and Hyperparameters ---
EMBEDDING_DIM = 10      # Dimension of the word embeddings
WINDOW_SIZE = 2         # Number of context words to consider on each side of the target word
LEARNING_RATE = 0.001   # Learning rate for the optimizer
NUM_EPOCHS = 50         # Number of training epochs

# --- Mock Dataset ---
# A small corpus to demonstrate word embedding learning.
# Notice some semantic relationships (e.g., 'natural language processing', 'deep learning', 'AI').
corpus = [
    "I love natural language processing",
    "natural language processing is a fascinating field",
    "PyTorch makes deep learning easy",
    "deep learning is a subset of machine learning",
    "machine learning and natural language processing are related",
    "AI is the future of technology",
    "PyTorch is a powerful framework for AI",
    "data science combines machine learning and statistics",
    "statistics is crucial for data analysis"
]

print("--- Dataset and Preprocessing ---")
print(f"Corpus size: {len(corpus)} sentences")

# --- Preprocessing: Tokenization and Vocabulary Building ---
words = []
for sentence in corpus:
    words.extend(sentence.lower().split())

word_counts = Counter(words)
# Filter out very rare words if desired, but for a small corpus, keep all.
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size}")
print(f"Sample word_to_idx: {list(word_to_idx.items())[:5]}...")
print(f"Sample idx_to_word: {list(idx_to_word.items())[:5]}...")

# --- Generate CBOW Training Data ---
# For CBOW, we predict a target word from its surrounding context words.
# Each data point will be (list_of_context_word_indices, target_word_index).
training_data = []
for sentence in corpus:
    tokens = [word_to_idx[word] for word in sentence.lower().split()]
    for i, target_word_idx in enumerate(tokens):
        context_words_indices = []
        # Collect context words within the window
        for j in range(max(0, i - WINDOW_SIZE), min(len(tokens), i + WINDOW_SIZE + 1)):
            if i != j: # Exclude the target word itself
                context_words_indices.append(tokens[j])
        
        if context_words_indices: # Only add if there are context words
            training_data.append((context_words_indices, target_word_idx))

print(f"Generated {len(training_data)} training pairs.")
print(f"Sample training pair: {training_data[0]} (Context indices, Target index)")
print(f"Corresponding words: {[idx_to_word[idx] for idx in training_data[0][0]]} -> {idx_to_word[training_data[0][1]]}")

# Note: For this exercise, we will iterate through `training_data` directly
# and handle context averaging within the model's forward pass for each sample.
# A more efficient batching for variable-length contexts would involve padding and masking,
# but that adds complexity not central to the embedding concept itself.


## Your Turn! Implement the CBOW Model and Training Loop

Now it's your turn to complete the exercise. Using the provided setup code (corpus, vocabulary, training data, and hyperparameters), implement the following:

1.  **Define the CBOW Model:**
    *   Create a PyTorch `nn.Module` class named `CBOWModel`.
    *   It should have an `nn.Embedding` layer and a `nn.Linear` layer.
    *   The `forward` method should take a tensor of context word indices, retrieve their embeddings, average them, and then pass the result through the linear layer to produce logits for the target word. Remember to apply `log_softmax` for `nn.NLLLoss`.

2.  **Instantiate Model, Loss, and Optimizer:**
    *   Create an instance of your `CBOWModel`.
    *   Define the loss function (e.g., `nn.NLLLoss`).
    *   Define the optimizer (e.g., `optim.Adam`).

3.  **Implement the Training Loop:**
    *   Iterate for `NUM_EPOCHS`.
    *   Inside each epoch, iterate through your `training_data`.
    *   For each `(context_indices, target_idx)` pair:
        *   Zero the gradients.
        *   Convert `context_indices` and `target_idx` to PyTorch tensors.
        *   Perform a forward pass through your model.
        *   Calculate the loss.
        *   Perform a backward pass (`loss.backward()`).
        *   Update model parameters (`optimizer.step()`).
        *   Print the loss periodically to monitor training progress.

4.  **Extract and Visualize Embeddings:**
    *   After training, retrieve the `weight` matrix from your model's `nn.Embedding` layer. This matrix contains your learned word embeddings.
    *   Use `sklearn.manifold.TSNE` (or `umap.UMAP` if you prefer and have it installed) to reduce the dimensionality of these embeddings to 2D.
    *   Plot the 2D embeddings using `matplotlib.pyplot.scatter`.
    *   Annotate each point with its corresponding word using `plt.annotate`.
    *   Add a title and labels to your plot.

Remember to leverage the `word_to_idx`, `idx_to_word`, `vocab_size`, `EMBEDDING_DIM`, `LEARNING_RATE`, `NUM_EPOCHS`, and `training_data` variables defined in the setup cell.


In [ ]:
# --- Reference Solution ---

# 1. Define the CBOW Model
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWModel, self).__init__()
        # The embedding layer maps word indices to dense vectors.
        # Each row in self.embeddings.weight corresponds to a word's embedding.
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # The linear layer projects the averaged context embedding back to the vocabulary size
        # to predict the target word.
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        # inputs: a tensor of shape (num_context_words,)
        # Get embeddings for all context words
        # embeds shape: (num_context_words, embedding_dim)
        embeds = self.embeddings(inputs)
        
        # Average the embeddings of the context words
        # avg_embeds shape: (embedding_dim,)
        avg_embeds = torch.mean(embeds, dim=0) # Average across the context words
        
        # Pass the averaged embedding through the linear layer
        # output shape: (vocab_size,)
        output = self.linear(avg_embeds)
        
        # Apply log_softmax for NLLLoss
        log_probs = torch.log_softmax(output, dim=0) # Apply across the vocabulary dimension
        return log_probs

# 2. Instantiate Model, Loss, and Optimizer
print("\n--- Model Initialization ---")
model = CBOWModel(vocab_size, EMBEDDING_DIM)
loss_function = nn.NLLLoss() # Negative Log Likelihood Loss is suitable with log_softmax output
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Model architecture:\n{model}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# 3. Implement the Training Loop
print("\n--- Training Started ---")
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    random.shuffle(training_data) # Shuffle data each epoch for better training
    
    for context_indices, target_idx in training_data:
        # Convert data to PyTorch tensors
        context_tensor = torch.tensor(context_indices, dtype=torch.long)
        target_tensor = torch.tensor(target_idx, dtype=torch.long)

        # Zero the gradients before running the backward pass
        model.zero_grad()

        # Forward pass: get log probabilities
        log_probs = model(context_tensor)

        # Compute loss
        # nn.NLLLoss expects input of shape (N, C) and target of shape (N) where N is batch size
        # Here, N=1, so we unsqueeze to add a batch dimension.
        loss = loss_function(log_probs.unsqueeze(0), target_tensor.unsqueeze(0))

        # Backward pass: compute gradients
        loss.backward()

        # Update model parameters
        optimizer.step()

        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0: # Print loss every 10 epochs
        print(f"Epoch {epoch + 1}/{NUM_EPOCHS}, Loss: {total_loss / len(training_data):.4f}")

print("--- Training Finished ---")

# 4. Extract Embeddings
# The learned embeddings are stored in the weight matrix of the nn.Embedding layer.
# We detach them from the computation graph and convert to numpy for visualization.
word_embeddings = model.embeddings.weight.data.cpu().numpy()
print(f"\nExtracted embeddings shape: {word_embeddings.shape}") # Should be (vocab_size, EMBEDDING_DIM)

# 5. Visualization using t-SNE
print("--- Visualizing Embeddings with t-SNE ---")

# Initialize t-SNE. perplexity is a crucial parameter, adjust based on dataset size.
# n_components=2 for 2D visualization.
# random_state for reproducibility.
# Perplexity must be less than the number of samples (vocab_size).
# A common heuristic is to keep it between 5 and 50. For very small vocab, adjust accordingly.
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, vocab_size - 1) if vocab_size > 1 else 1)

# Fit t-SNE to the embeddings and transform them to 2D
low_dim_embeddings = tsne.fit_transform(word_embeddings)

# Create the plot
plt.figure(figsize=(12, 10))
for i, word in idx_to_word.items():
    x, y = low_dim_embeddings[i, 0], low_dim_embeddings[i, 1]
    plt.scatter(x, y, color='blue')
    plt.annotate(word, xy=(x, y), xytext=(5, 2), textcoords='offset points',
                 ha='right', va='bottom', fontsize=9)

plt.title('2D Visualization of Word Embeddings (t-SNE)')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("\n--- Analysis of Visualization ---")
print("Observe the plot. Words that are semantically related (e.g., 'deep learning', 'machine learning', 'AI')")
print("should ideally appear closer to each other. Due to the small corpus and simple model,")
print("the relationships might not be perfectly clear, but some clusters might emerge.")
print("For larger, more complex datasets and models, these visualizations are powerful tools")
print("for understanding the learned representations.")

# Optional: For 2026 readiness, mention UMAP
# UMAP (Uniform Manifold Approximation and Projection) is another powerful
# dimensionality reduction technique often preferred over t-SNE for its speed
# and ability to preserve global structure better. To use UMAP, you would install
# 'umap-learn' (pip install umap-learn) and then:
# import umap
# reducer = umap.UMAP(n_components=2, random_state=42)
# low_dim_embeddings_umap = reducer.fit_transform(word_embeddings)
# Then plot similarly to the t-SNE example.
